## OpenMeteo API

In [ ]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry
from timezonefinder import TimezoneFinder



In [ ]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 10)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
lat, lng = -22.934, -43.180
tz = TimezoneFinder().certain_timezone_at(lat=lat, lng=lng)

params = {
	"latitude": lat,
	"longitude": lng,
	"hourly": ["temperature_2m", "precipitation_probability", "precipitation"],
    "forecast_hours" : 6,
	"current": ["temperature_2m", "precipitation", "wind_speed_10m", "wind_direction_10m"],
}
responses = openmeteo.weather_api(url, params=params)


In [ ]:
from numpy import round

def get_cardinal_wind_direction(degrees:float) -> str:
    directions = ['N ↓', 'NE ↙', 'E ←', 'SE ↖', 'S ↑', 'SW ↗', 'W →', 'NW ↘']
    idx = round(degrees / 45) % 8
    return directions[int(idx)]

def get_lat_lng_str(coordinates:float,type:str) -> str:
    if not type in ["lat", "lng"]:
        raise ValueError("Type must be 'lat' or 'lng'")
    if type == "lat":
        return f"{abs(coordinates)}°S" if coordinates < 0 else f"{coordinates}°N"
    else:
        return f"{abs(coordinates)}°W" if coordinates < 0 else f"{coordinates}°E"

In [ ]:
response = responses[0]
print(f"Coordinates: {get_lat_lng_str(response.Latitude(), 'lat')} {get_lat_lng_str(response.Longitude(), 'lng')}")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {tz}")

# Process current data. The order of variables needs to be the same as requested.
current = response.Current()
current_temperature_2m = current.Variables(0).Value()
current_precipitation = current.Variables(1).Value()
current_wind_speed_10m = current.Variables(2).Value()
current_wind_direction_10m = current.Variables(3).Value()

print(f"\nCurrent Weather Update: { pd.to_datetime(current.Time(), unit = 's', utc = True).tz_convert(tz) }")
print(f"Current temperature_2m: {int(round(current_temperature_2m, 1))} °C")
print(f"Current precipitation: {current_precipitation} mm")
print(f"Current wind_speed_10m: {current_wind_speed_10m} m/s")
print(f"Current wind_direction_10m: {get_cardinal_wind_direction(current_wind_direction_10m)} ({current_wind_direction_10m}°)")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True).tz_convert(tz),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True).tz_convert(tz),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["precipitation"] = hourly_precipitation

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)
    

## HGBrasil API 

In [ ]:
import requests
from dotenv import load_dotenv
from os import getenv

load_dotenv()
api_key = getenv('HG_BRAZIL_API_KEY')
lat, lon = -22.934, -43.180

url = 'https://api.hgbrasil.com/weather'

params = {
    'lat': lat,
    'lon': lon,
    'key': api_key
}

response = requests.get(url, params=params)
data = response.json()

In [ ]:
data.keys()

In [ ]:

forecast = data.get("results", {}).get("forecast", [])
print(forecast)

## Meteoblue API 

In [ ]:
import requests
from dotenv import load_dotenv
from os import getenv


load_dotenv()

api_key = getenv('METEO_BLUE_API_KEY')
lat, lon = -22.934, -43.180
url = f"https://my.meteoblue.com/packages/current_basic-1h_basic-day?lat={lat}&lon={lon}&apikey={api_key}&forecast_days=1"

response = requests.get(url)
data = response.json()

### Saving request data to disk

In [ ]:
import json

with open("meteo_blue_response.json", "w") as f:
    json.dump(data, f, indent=4)

### Load saved data to avoid spending API call credits

In [ ]:
with open("meteo_blue_response.json", "r") as f:
    data = json.load(f)

In [ ]:
import pandas as pd

tz = data.get("metadata",{}).get("timezone_abbrevation","utc")
if tz.upper()=="UTC":
    tz = "UTC"
else:
    tz = f"Etc/{tz.replace("0","")}"

current = data.get("data_current", {})
predict = pd.DataFrame(data.get("data_1h", []))

In [ ]:
predict["winddirection_cardinal"] = predict["winddirection"].apply(get_cardinal_wind_direction)
predict = predict[[
    'time',
    'windspeed',
    'winddirection',
    'winddirection_cardinal',
    'temperature',
    'precipitation_probability',
    'precipitation',
]]
predict


# Transformations

## generic

In [ ]:
import time
from numpy import nan as np_nan
import pandas as pd


def isnull(value):
    return pd.isnull(value) or value == {} or value == tuple() or value == [] or value == [{}]

test_values = [[], [{}], {}, tuple(), '', None, np_nan, [1,2], {"a":1}, (1,)]
for val in test_values:
    print(f"Value: {val!r}, isnull: {isnull(val)}")



In [ ]:
def transform_and_clean_data(
        data: pd.DataFrame,
        transform_function: callable = None,
        parameters:dict = None,
        target_fields: list[str] = None,
        clean: bool = False
        ) -> pd.DataFrame:
    if data.empty:
        print("No data to process, returning empty DataFrame.")
        return data

    print("Starting to process %s rows", len(data))
    start_time = time.time()

    if transform_function:
        data = transform_function(data, parameters)

    if target_fields:
        data = data[target_fields]

    if clean:
        data = data.replace({pd.NA: None, pd.NaT: None, np_nan: None, })

    end_time = time.time()
    elapsed_time = time.strftime("%H:%M:%S", time.gmtime(end_time - start_time))
    print("Done processing.\nTotal rows after processing: %s\nElapsed time: %s", len(data), elapsed_time)

    return data

df_example = pd.DataFrame({
    "col1": [1, 2, None, np_nan],
    "col2": ["a", "", "c", "d"],
})


def sample_transform(df, params):
    df["col1"] = df["col1"].apply(lambda x: x * 10 if not isnull(x) else 0)    
    return df

# Testando transform_and_clean_data
df = transform_and_clean_data(
    df_example,
    transform_function=sample_transform,
    parameters=None,
    target_fields=["col1", "col2", "col3"],
    clean=True
)

print("\nDataFrame resultante:")
print(df)

## json

In [ ]:
import pandas as pd

def isnull(value):
    if isinstance(value, list):
        if len(value) == 0 or value == [{}]:
            return True
        else:
            return False
    if pd.isnull(value) or value == '' or value == {} or value == tuple():
        return True
    return False

df = pd.DataFrame({
    'A': [1, 2, 3], 
    'B': [True, False, False], 
    'C': ['a', 'b', 'c'],
    'D': [{'e': 4, 'f': {'g':7,'h': 9}}, None, {'e': 6, 'f':{'g': 8, 'h': 10}}]
})

mapping = {
    'h':('D',('f','h',)),
    'g':('D',('f','g',)),
    'e':('D',('e',)),
}
def check_and_load_json_obj(value: str|dict|None) -> dict|None:
    if isinstance(value, str):
        return json.loads(value)
    return value

def get_nested_json_data(json_obj: dict, path: tuple):
    print(f"Accessing path {path} in JSON object: {json_obj}")
    if isnull(json_obj):
        return None
    if isnull(path):
        return json_obj
    if not isinstance(json_obj, (dict, list)):
        return None
    key = path[0]
    try:
        return get_nested_json_data(json_obj[key], path[1:])
    except:
        print(f"Error accessing key '{key}' in JSON object: {json_obj}")

def unnest_json_columns(data: pd.DataFrame, mapping: dict[str, tuple[str, tuple]]) -> pd.DataFrame:
    for col in set(df_col for df_col, _ in mapping.values()):
        data[col] = data[col].apply(check_and_load_json_obj)
    for new_col, (df_col, path) in mapping.items():
        data[new_col] = data[df_col].apply(lambda x: get_nested_json_data(x, path))
    return data

df = unnest_json_columns(df, mapping)
print(df)


# Pipeline Testing

In [85]:
import pandas as pd
import json

with open("meteo_blue_response.json", "r") as f:
    json_data = json.load(f)

data = pd.DataFrame([json_data])
data_1h = pd.DataFrame()
data_1h["time"] = data["data_1h"].apply(lambda x: x.get("time"))
data_1h["temp"] = data["data_1h"].apply(lambda x: x.get("temperature"))
data_1h = data_1h.explode(["time","temp"]).reset_index(drop=True)
print(data_1h)


                time   temp
0   2026-03-04 00:00   25.0
1   2026-03-04 01:00   25.0
2   2026-03-04 02:00  24.67
3   2026-03-04 03:00   24.0
4   2026-03-04 04:00   23.0
5   2026-03-04 05:00  23.67
6   2026-03-04 06:00  24.33
7   2026-03-04 07:00   25.0
8   2026-03-04 08:00  25.67
9   2026-03-04 09:00  26.33
10  2026-03-04 10:00   27.0
11  2026-03-04 11:00   28.0
12  2026-03-04 12:00   28.5
13  2026-03-04 13:00  28.71
14  2026-03-04 14:00  28.75
15  2026-03-04 15:00   28.6
16  2026-03-04 16:00  28.12
17  2026-03-04 17:00  27.47
18  2026-03-04 18:00  26.74
19  2026-03-04 19:00  26.29
20  2026-03-04 20:00  26.07
21  2026-03-04 21:00  25.98
22  2026-03-04 22:00  25.72
23  2026-03-04 23:00  25.41
24  2026-03-05 00:00  24.98


In [77]:
import pandas as pd

# Exemplo de DataFrame
df = pd.DataFrame({
    "A": [1, 2, 3],
    "B": [[{"x":1, "y":2}, {"x":3, "y":4}], [{"x":5, "y":6}, {"x":7, "y":8}], [{"x":9, "y":10}, {"x":11, "y":12}]],
    "C": [10, 20, 30]
})

print("Original df:")
print(df)

# Detach column B into a new DataFrame
df_B = pd.DataFrame(df.pop("B"))
df_B = df_B.explode("B").reset_index(drop=True)

print("\nNew df_B (detached column):")
print(df_B)

print("\nModified df (after detaching B):")
print(df)

Original df:
   A                                        B   C
0  1     [{'x': 1, 'y': 2}, {'x': 3, 'y': 4}]  10
1  2     [{'x': 5, 'y': 6}, {'x': 7, 'y': 8}]  20
2  3  [{'x': 9, 'y': 10}, {'x': 11, 'y': 12}]  30

New df_B (detached column):
                    B
0    {'x': 1, 'y': 2}
1    {'x': 3, 'y': 4}
2    {'x': 5, 'y': 6}
3    {'x': 7, 'y': 8}
4   {'x': 9, 'y': 10}
5  {'x': 11, 'y': 12}

Modified df (after detaching B):
   A   C
0  1  10
1  2  20
2  3  30
